# 1 — Snowflake Notebooks
## "Your Jupyter, but in Snowflake"

This is the same notebook interface you already use. The difference is *where the work happens*.

Today the workflow is:

> query Snowflake &rarr; **download the result** &rarr; process it in pandas on a laptop &rarr; write results back

That download step is the bottleneck. It caps you at local RAM, it means every teammate needs an identical Python environment, and it puts patient-adjacent data onto endpoints.

Here the download step is gone. SQL and Python cells sit side by side, both executing on Snowflake compute, against data that never leaves the account.

**Dataset:** synthetic liquid-biopsy cohort — 50,000 patients, 120,000 blood draws, **20,000,000 raw variant calls**. None of it is real patient data.

### Set the context

No `pip install`, no conda environment, no credentials file. The notebook already knows who you are and what you are allowed to see.

In [ ]:
USE WAREHOUSE GUARDANT_DEMO_WH;
USE SCHEMA DEMO.GUARDANT_DEMO;

SELECT CURRENT_USER()      AS whoami,
       CURRENT_ROLE()      AS acting_as,
       CURRENT_WAREHOUSE() AS compute,
       CURRENT_SCHEMA()    AS working_in;

### How big is the table?

This is the number that matters. 20 million raw calls is one modest cohort, and it is already past the point where `pd.read_sql(...)` on a laptop is a good idea.

In [ ]:
SELECT COUNT(*)                                         AS raw_variant_calls,
       COUNT(DISTINCT specimen_id)                      AS specimens,
       COUNT(DISTINCT gene_symbol)                      AS genes_on_panel,
       ROUND(COUNT(*) / COUNT(DISTINCT specimen_id), 1) AS avg_calls_per_specimen
FROM VARIANT_CALLS;

### The filtering step you run every day

Raw calls include the noise floor: low mapping quality, shallow depth, sub-threshold VAF. Filtering it out is the first thing any variant analysis does.

On a laptop that means pulling all 20M rows down and *then* discarding most of them. Here the filter runs where the data already lives, and only the summary comes back.

In [ ]:
SELECT call_filter,
       COUNT(*)                                           AS calls,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM VARIANT_CALLS
GROUP BY call_filter
ORDER BY calls DESC;

### Hand a SQL result straight to Python

This is the part that tends to surprise people. Any SQL cell is addressable from Python **by its cell name** — `sql_gene_burden.to_pandas()`. No connector, no cursor, no credentials.

The aggregation ran on Snowflake across 20M rows. What crosses into Python is 15 rows.

In [ ]:
SELECT gene_symbol,
       COUNT(*)                    AS reportable_calls,
       COUNT(DISTINCT specimen_id) AS specimens_affected,
       ROUND(MEDIAN(vaf) * 100, 3) AS median_vaf_pct,
       ROUND(AVG(read_depth))      AS mean_depth,
       MAX(is_actionable)          AS has_targeted_therapy
FROM V_REPORTABLE_VARIANTS
GROUP BY gene_symbol
ORDER BY reportable_calls DESC
LIMIT 15;

In [ ]:
# The SQL cell above is available here by name. One method call.
df = sql_gene_burden.to_pandas()

print(f"Rows that crossed into Python: {len(df):,}")
print(f"Rows scanned on Snowflake:     20,000,000")
df

### Plot it with the libraries you already use

matplotlib, seaborn, plotly, scipy, scikit-learn — all available from the packages picker. Charts render inline.

In [ ]:
import matplotlib.pyplot as plt

plot_df = df.sort_values("REPORTABLE_CALLS")
colors = ["#29B5E8" if a else "#B0BEC5" for a in plot_df["HAS_TARGETED_THERAPY"]]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_df["GENE_SYMBOL"], plot_df["REPORTABLE_CALLS"], color=colors)
ax.set_xlabel("Reportable variant calls")
ax.set_title("Mutation burden by gene\n(blue = targeted therapy available)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Or skip SQL entirely and stay in Python

If you would rather express the whole thing as a DataFrame, you can. `session.table(...)` returns a **lazy** Snowpark DataFrame: the operations below assemble a query, and nothing executes until you ask for a result.

That laziness is the bridge into notebook 2.

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()

variants = session.table("V_REPORTABLE_VARIANTS")

# Nothing has executed yet — this is a query being assembled, not data being moved.
actionable_by_cancer = (
    variants
    .filter(F.col("IS_ACTIONABLE") & (F.col("VAF") >= 0.05))
    .group_by("PRIMARY_CANCER_TYPE")
    .agg(
        F.count_distinct("PATIENT_ID").alias("PATIENTS_WITH_ACTIONABLE"),
        F.count("*").alias("ACTIONABLE_CALLS"),
    )
    .sort(F.col("PATIENTS_WITH_ACTIONABLE").desc())
)

actionable_by_cancer.show(10)

---

### What changed

| | Today | Here |
|---|---|---|
| Where compute runs | Your laptop | Snowflake |
| Data movement | Full extract, every time | Results only |
| Environment setup | Per person, per machine | None |
| Ceiling | Local RAM | Warehouse size |
| Governance | Data on endpoints | Never leaves the account |

Next: **notebook 2** takes the same data to a scale with no laptop equivalent.